# 02 · Extract fields with `ai_extract`

Feed the materialized `parsed_text` into **`ai_extract`** to pull the certificate fields.
`ai_extract` works on **already-parsed text** (it can't read PDF bytes — that's why
`ai_parse_document` runs first). We use **v2.1** with **per-field citations + confidence**
so low-confidence fields can be routed to a human reviewer.

In [0]:
CATALOG, SCHEMA = "users", "scott_mckean"
import json

FIELDS = {
    "document_type": "The type of insurance document, e.g. Certificate of Insurance, Policy Declaration, Endorsement",
    "insurer_name": "The name of the insurance company providing coverage",
    "insured_name": "The full legal name of the person or entity covered by the policy",
    "policy_number": "The unique alphanumeric identifier assigned to the insurance policy",
    "effective_date": "The date coverage begins, in ISO 8601 format (YYYY-MM-DD)",
    "expiration_date": "The date coverage ends, in ISO 8601 format (YYYY-MM-DD)",
    "coverage_types": "Comma-separated list of coverage types, e.g. General Liability, Auto Liability, Workers Compensation",
}

## 1 · Extract → materialized `extracted_fields` / `extracted_flat`

In [0]:
# Build object-typed schema required by ai_extract v2.1 with citations/confidence
schema_json = json.dumps({k: {"type": "string", "description": v} for k, v in FIELDS.items()})

spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.extracted_fields AS
    SELECT filename,
           ai_extract(full_text, '{schema_json}',
               options => map('version', '2.1',
                              'enableCitations', 'true',
                              'enableConfidenceScores', 'true')) AS extracted
    FROM {CATALOG}.{SCHEMA}.parsed_text
""")

value_cols = ',\n'.join([f"extracted:response:{f}:value::string AS {f}" for f in FIELDS])
conf_cols  = ',\n'.join([f"extracted:response:{f}:confidence_score::double AS {f}_conf" for f in FIELDS])
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.extracted_flat AS
    SELECT filename, {value_cols}, {conf_cols}
    FROM {CATALOG}.{SCHEMA}.extracted_fields
""")
print("Extracted:", spark.table(f"{CATALOG}.{SCHEMA}.extracted_flat").count(), "docs")
display(spark.sql(f"""
    SELECT filename, document_type, insurer_name, policy_number,
           effective_date, expiration_date, substr(coverage_types,1,50) AS coverage_types
    FROM {CATALOG}.{SCHEMA}.extracted_flat
"""))

## 2 · Human-in-the-loop review queue

Confidence scores let you auto-accept high-confidence extractions and route the rest to a
reviewer — the straight-through-processing pattern.

In [0]:
THRESH = 0.80

# Use raw confidence columns — don't coalesce NULLs to 0
conf_cols_raw = ", ".join([f"{f}_conf" for f in FIELDS])

# Count NULLs per row to detect missing extractions
null_count_expr = " + ".join([f"CASE WHEN {f}_conf IS NULL THEN 1 ELSE 0 END" for f in FIELDS])

review = spark.sql(f"""
    SELECT filename,
           ({null_count_expr}) AS missing_fields,
           round(least({conf_cols_raw}), 3) AS min_field_confidence,
           CASE
               WHEN ({null_count_expr}) > 0 THEN 'missing_extraction'
               ELSE 'low_confidence'
           END AS review_reason
    FROM {CATALOG}.{SCHEMA}.extracted_flat
    WHERE least({conf_cols_raw}) < {THRESH}
       OR ({null_count_expr}) > 0
    ORDER BY missing_fields DESC, min_field_confidence
""")
print(f"{review.count()} docs need review (threshold={THRESH})")
display(review)

_Next: `03_evaluate_mlflow`._